In [ ]:
import time
from openai import OpenAI
import math
import os
from openai import OpenAI
import json
import pandas as pd
from tqdm import tqdm

client = OpenAI(
    api_key="OPENAI_API_KEY",
    base_url="OPENAI_BASE_URL",
)
TEMPERATURE = 0.1
LLM_MODEL = "gpt-5.4"
input_file = "patient_data.xlsx"
output_file = "patient_data_cleaned_reports"

def fetch_response(content, max_retries=10, sleep_duration=5):
    retries = 0
    while retries < max_retries:
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "user", "content": content}
                ],
                model=LLM_MODEL,
                temperature=TEMPERATURE,
            )
            result = response.choices[0].message.content
            return result

        except Exception as e:
            print(f"Error during API call: {str(e)}")
            retries += 1
            time.sleep(sleep_duration)
            print(f"Retry {retries}/{max_retries}...")

        if retries == max_retries:
            raise Exception("Max retries reached, failing...")
    return None

In [2]:
SYSTEM_PROMPT = """
You are a clinical text de-identification and label-leakage cleaning assistant.

Task:
Clean the input report by removing only direct disease names, explicit diagnoses, and prognosis/outcome statements.

Strict rules:
1. Preserve all other wording, punctuation, capitalization, spacing, order, and measurement values exactly as much as possible.
2. Do not summarize, rewrite, normalize, translate, or add any text.
3. Remove direct disease names and diagnostic labels, including specific tumor entities, cancer entities, syndrome names, and explicit diagnostic conclusions after labels such as Clinical:, Diagnosis:, Impression:, Biopsy:, Histopathology:, Cytology:, Pathology:, or similar.
4. Do not delete generic descriptive words by themselves, such as tumor, mass, lesion, nodule, infiltrate, neoplasm, thickening, dysplasia, atypical, malignant, positive, or negative, unless they are part of a direct disease name or explicit diagnosis that must be removed.
5. Keep non-diagnostic clinical findings, examination findings, imaging findings, laboratory findings, procedures, stains, biomarkers, genes, variants, measurements, anatomic sites, and descriptive morphology when they are not themselves a direct disease name or prognosis.
6. Keep infectious or inflammatory test findings such as fungal hyphae, fungal filaments, organisms, bacteria, viral inclusions, inflammatory cells, conjunctivitis, dacryoadenitis, keratitis, or ulcer findings unless the phrase is an explicit final diagnosis or prognosis.
7. Remove prognosis/outcome information such as survival, death, remission, recurrence, progression, relapse, metastasis status, disease-free status, worsening, improvement, response, follow-up outcome, and final outcome.
8. When removing an explicit diagnosis item, remove the whole semicolon-delimited item if needed so that no empty labels, dangling colons, double semicolons, or meaningless fragments remain.
9. Return only the cleaned report text. Do not return explanations, markdown, quotes, or JSON.
"""

CHECKPOINT_FILE = output_file + "_checkpoint.xlsx"
FINAL_OUTPUT_FILE = output_file + ".xlsx"
DONE_COL = "__cleaned_reports_done"
SAVE_EVERY = 50


def clean_report_text(report_text, row_max_retries=3, sleep_duration=2):
    if pd.isna(report_text):
        return report_text

    original_text = str(report_text)
    if original_text.strip() == "":
        return original_text

    content = f"{SYSTEM_PROMPT}\n\nReport text:\n{original_text}"
    last_error = None

    for attempt in range(1, row_max_retries + 1):
        try:
            cleaned_text = fetch_response(content, max_retries=10, sleep_duration=sleep_duration)
            cleaned_text = cleaned_text.strip('\n')
            if cleaned_text.startswith('```') and cleaned_text.endswith('```'):
                cleaned_text = cleaned_text.strip('`').strip('\n')
            if not is_deletion_only(original_text, cleaned_text):
                raise ValueError("Model output changed or added text instead of only deleting text.")
            return cleaned_text
        except Exception as e:
            last_error = e
            print(f"Row-level retry {attempt}/{row_max_retries} failed: {e}")
            time.sleep(sleep_duration * attempt)

    raise last_error


def is_deletion_only(original_text, cleaned_text):
    """Return True only if cleaned_text can be made by deleting chars from original_text."""
    if len(cleaned_text) > len(original_text):
        return False

    pos = 0
    for ch in cleaned_text:
        pos = original_text.find(ch, pos)
        if pos == -1:
            return False
        pos += 1
    return True


def save_checkpoint(df):
    df.to_excel(CHECKPOINT_FILE, index=False)
    print(f"Checkpoint saved: {CHECKPOINT_FILE}")


if os.path.exists(CHECKPOINT_FILE):
    print(f"Resuming from checkpoint: {CHECKPOINT_FILE}")
    df = pd.read_excel(CHECKPOINT_FILE)
else:
    df = pd.read_excel(input_file)
    df[DONE_COL] = False

if "reports" not in df.columns:
    raise ValueError("Column 'reports' not found in input file.")

if DONE_COL not in df.columns:
    df[DONE_COL] = False

pending_idx = df.index[~df[DONE_COL].fillna(False).astype(bool)].tolist()
print(f"Total rows: {len(df)}; pending rows: {len(pending_idx)}")

try:
    for n, idx in enumerate(tqdm(pending_idx, desc="Cleaning reports", unit="row"), start=1):
        df.loc[idx, "reports"] = clean_report_text(df.loc[idx, "reports"])
        df.loc[idx, DONE_COL] = True

        if n % SAVE_EVERY == 0:
            save_checkpoint(df)

except KeyboardInterrupt:
    save_checkpoint(df)
    print("Interrupted. Re-run this cell to continue from the checkpoint.")
    raise
except Exception:
    save_checkpoint(df)
    print("Error encountered. Re-run this cell to continue from the checkpoint after fixing the issue.")
    raise

save_checkpoint(df)
df.drop(columns=[DONE_COL]).to_excel(FINAL_OUTPUT_FILE, index=False)
print(f"Finished. Final output saved: {FINAL_OUTPUT_FILE}")

Total rows: 2134; pending rows: 2134


Cleaning reports:   2%|▏         | 50/2134 [02:51<2:27:23,  4.24s/row]

Checkpoint saved: patient_data_v5_cleaned_reports_checkpoint.xlsx


Cleaning reports:   5%|▍         | 100/2134 [05:56<3:28:19,  6.15s/row]

Checkpoint saved: patient_data_v5_cleaned_reports_checkpoint.xlsx


Cleaning reports:   7%|▋         | 150/2134 [08:20<1:35:07,  2.88s/row]

... (83 outputs left out)

Cleaning reports:  98%|█████████▊| 2100/2134 [2:21:40<05:16,  9.32s/row]

Checkpoint saved: patient_data_v5_cleaned_reports_checkpoint.xlsx


Cleaning reports: 100%|██████████| 2134/2134 [2:23:44<00:00,  4.04s/row]


Checkpoint saved: patient_data_v5_cleaned_reports_checkpoint.xlsx
Finished. Final output saved: patient_data_v5_cleaned_reports.xlsx
